# 🏭 AIoT Cold Chain Anomaly Detection Lab
### วิเคราะห์และตรวจจับความผิดปกติของตู้แช่ยาและห้อง Server ด้วย Machine Learning
**วิชา:** CN466 Project - Internet of Things & AI  
**ผู้พัฒนา:** Chanapa Sadudee (6610680214)  

---
### 🎯 วัตถุประสงค์การเรียนรู้
1. เข้าใจโครงสร้างข้อมูล Telemetry (อุณหภูมิและความชื้น) จากบอร์ด Cucumber
2. เรียนรู้วิธีสกัด Features เชิงพยากรณ์ เช่น Rolling Slope และ Z-Score
3. เทรนโมเดล **Isolation Forest** และประเมินจุดวิกฤตความเย็นล้มเหลว (Cooling Failure)

In [ ]:
# 1. ติดตั้งและนำเข้าไลบรารีที่จำเป็น
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

print('✅ Libraries loaded successfully!')

### 2. จำลองข้อมูลเซ็นเซอร์ (Synthetic Cold Chain Telemetry)
จำลองข้อมูล 600 วินาที: ช่วงแรกปกติ (4°C) และเกิดวิกฤตคอมเพรสเซอร์ดับช่วงวินาทีที่ 350

In [ ]:
np.random.seed(42)
n_steps = 600
timestamps = pd.date_range('2026-11-14 08:00:00', periods=n_steps, freq='2s')

# สภาวะปกติ: 4.2°C +/- noise
temp = 4.2 + np.random.normal(0, 0.15, n_steps)
humid = 60.0 + np.random.normal(0, 0.8, n_steps)

# จำลองวิกฤตตู้แช่ดับตั้งแต่ step 350 เป็นต้นไป
for i in range(350, n_steps):
    temp[i] = temp[i-1] + np.random.uniform(0.08, 0.18)
    humid[i] = humid[i-1] + np.random.uniform(0.1, 0.3)

df = pd.DataFrame({'timestamp': timestamps, 'temp': np.round(temp, 2), 'humid': np.round(humid, 2)})
df.head()

### 3. Feature Engineering: Rolling Slope & Z-Score
การตรวจจับความผิดปกติที่ดีต้องดู **อัตราการเปลี่ยนแปลง (Rate of Change)** ไม่ใช่แค่รอให้อุณหภูมิเกิน 8°C

In [ ]:
# คำนวณ Moving Average และ Slope
df['temp_roll_mean'] = df['temp'].rolling(window=10).mean()
df['temp_roll_std'] = df['temp'].rolling(window=10).std()
df['temp_slope'] = df['temp'].diff(periods=5) / 5.0
df['z_score'] = (df['temp'] - df['temp_roll_mean']) / (df['temp_roll_std'] + 1e-5)
df.dropna(inplace=True)
df.head()

### 4. เทรนโมเดล Machine Learning (Isolation Forest)

In [ ]:
features = df[['temp', 'humid', 'temp_slope', 'z_score']]
model = IsolationForest(contamination=0.15, random_state=42)
df['anomaly'] = model.fit_predict(features)
df['is_anomaly'] = df['anomaly'] == -1

print(f"จำนวนจุดที่พบความผิดปกติ: {df['is_anomaly'].sum()} จากทั้งหมด {len(df)} จุด")

### 5. แสดงผลกราฟการตรวจจับความผิดปกติ

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df['timestamp'], df['temp'], label='Temperature (°C)', color='#0284c7', lw=1.5)
plt.axhline(8.0, color='red', linestyle='--', alpha=0.7, label='GMP Safe Limit (8°C)')
plt.axhline(2.0, color='blue', linestyle='--', alpha=0.7, label='Lower Limit (2°C)')

# ไฮไลต์จุดที่ AI ตรวจพบความผิดปกติ
anomalies = df[df['is_anomaly']]
plt.scatter(anomalies['timestamp'], anomalies['temp'], color='crimson', s=20, label='AI Detected Anomaly')

plt.title('Cold Chain Anomaly Detection with Isolation Forest and Predictive Slope')
plt.xlabel('Time')
plt.ylabel('Temp (°C)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()